# Great Expectations (GX) – Bronze suite (monitoring)

Cilj: definirati i pokrenuti data quality provjere nad BRONZE slojem (dbt view) bez mijenjanja podataka.

Dataset/tablica (dbt output):
- Project: tpiuo-labosi
- Dataset: stackoverflow_dbt
- Model (view): stackoverflow_questions_bronze

Napomena: Bronze provjere služe za monitoring (schema drift, volumen, NULL udio, duplikati).


In [3]:
import os
import json
from datetime import datetime, timezone

from google.cloud import bigquery
import great_expectations as gx

PROJECT_ID = "tpiuo-labosi"
DBT_DATASET = "stackoverflow_dbt"
BRONZE_TABLE = f"{PROJECT_ID}.{DBT_DATASET}.stackoverflow_questions_bronze"

print("GX version:", gx.__version__)

GX version: 1.11.0


In [4]:
bq = bigquery.Client(project=PROJECT_ID)

bronze_df = bq.query(f"SELECT * FROM `{BRONZE_TABLE}`").to_dataframe()
bronze_df.head(), bronze_df.shape

(   question_id                                              title  \
 0     79859107    i cant install taiwind css in my django project   
 1     79854178          googlesearch module in python not working   
 2     79858268  django model with foreign key onto settings.AU...   
 3     79858365  Propagating logging.logger from module to sing...   
 4     79857831       AttributeError on &#39;mediapipe&#39; module   
 
                                                 link  creation_date  \
 0  https://stackoverflow.com/questions/79859107/i...     1767354049   
 1  https://stackoverflow.com/questions/79854178/g...     1766560210   
 2  https://stackoverflow.com/questions/79858268/d...     1767196840   
 3  https://stackoverflow.com/questions/79858365/p...     1767212567   
 4  https://stackoverflow.com/questions/79857831/a...     1767131513   
 
    last_activity_date  is_answered  score  answer_count  view_count  \
 0          1767354049        False     -7             0          23   


In [5]:
context = gx.get_context(mode="file")  # file-based context stored in project folder

# Pandas datasource
try:
    ds = context.data_sources.get("so_pandas")
except Exception:
    ds = context.data_sources.add_pandas(name="so_pandas")

# Dataframe asset + batch definition
try:
    asset = ds.get_asset("bronze_questions")
except Exception:
    asset = ds.add_dataframe_asset(name="bronze_questions")

try:
    bronze_batch_def = asset.get_batch_definition("whole_df_bronze")
except Exception:
    bronze_batch_def = asset.add_batch_definition_whole_dataframe("whole_df_bronze")

bronze_batch_def


BatchDefinition(id=UUID('2ba978e8-428e-4cc0-83e2-a1f4a9827bf2'), name='whole_df_bronze', partitioner=None)

In [6]:
suite_name = "stackoverflow_bronze_suite"

try:
    bronze_suite = context.suites.get(suite_name)
except Exception:
    bronze_suite = gx.ExpectationSuite(name=suite_name)
    bronze_suite = context.suites.add(bronze_suite)

ex = gx.expectations

# Schema drift monitoring (columns exist)
bronze_suite.add_expectation(ex.ExpectColumnToExist(column="question_id"))
bronze_suite.add_expectation(ex.ExpectColumnToExist(column="creation_date"))
bronze_suite.add_expectation(ex.ExpectColumnToExist(column="title"))

# Volume
bronze_suite.add_expectation(ex.ExpectTableRowCountToBeBetween(min_value=1, max_value=10_000_000))

# Uniqueness monitoring (should be unique; if you expect duplicates, switch to mostly or remove)
bronze_suite.add_expectation(ex.ExpectColumnValuesToBeUnique(column="question_id"))

# NULL monitoring with mostly (allow up to 20% NULL)
bronze_suite.add_expectation(ex.ExpectColumnValuesToNotBeNull(column="creation_date", mostly=0.8))
bronze_suite.add_expectation(ex.ExpectColumnValuesToNotBeNull(column="title", mostly=0.8))

bronze_suite

{
  "name": "stackoverflow_bronze_suite",
  "id": "789b2da4-2168-4ec7-8080-b38a91f217b4",
  "expectations": [
    {
      "type": "expect_column_to_exist",
      "kwargs": {
        "column": "question_id"
      },
      "meta": {},
      "id": "239631b8-01fe-4256-9425-1f620abe5888",
      "severity": "critical"
    },
    {
      "type": "expect_column_to_exist",
      "kwargs": {
        "column": "creation_date"
      },
      "meta": {},
      "id": "fe369e33-206f-404c-9f1c-1438302a8c3f",
      "severity": "critical"
    },
    {
      "type": "expect_column_to_exist",
      "kwargs": {
        "column": "title"
      },
      "meta": {},
      "id": "99b472e5-7cb4-473a-ad1d-cc8f499be34f",
      "severity": "critical"
    },
    {
      "type": "expect_table_row_count_to_be_between",
      "kwargs": {
        "min_value": 1,
        "max_value": 10000000
      },
      "meta": {},
      "id": "f5d78306-a946-4aad-a8ae-62cb33d3145f",
      "severity": "critical"
    },
    {
      "t

In [7]:
ts = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
validation_name = f"bronze_validation_{ts}"

vdef = gx.ValidationDefinition(data=bronze_batch_def, suite=bronze_suite, name=validation_name)
bronze_result = vdef.run(batch_parameters={"dataframe": bronze_df})

bronze_result.success

Calculating Metrics: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████| 21/21 [00:00<00:00, 1846.16it/s]


True

In [9]:
os.makedirs("gx_results", exist_ok=True)
out_path = f"gx_results/{validation_name}.json"

with open(out_path, "w", encoding="utf-8") as f:
    json.dump(bronze_result.to_json_dict(), f, ensure_ascii=False, indent=2)

out_path

'gx_results/bronze_validation_20260120T173035Z.json'